In [1]:
import os
import sys
SPARK_HOME = "/usr/lib/spark3"
PYSPARK_PYTHON = "/opt/conda/envs/dsenv/bin/python"
os.environ["PYSPARK_PYTHON"]= PYSPARK_PYTHON
os.environ["PYSPARK_DRIVER_PYTHON"]= PYSPARK_PYTHON
os.environ["SPARK_HOME"] = SPARK_HOME

PYSPARK_HOME = os.path.join(SPARK_HOME, "python/lib")
sys.path.insert(0, os.path.join(PYSPARK_HOME, "py4j-0.10.9.7-src.zip"))
sys.path.insert(0, os.path.join(PYSPARK_HOME, "pyspark.zip"))

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, HashingTF
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator 

raw_data = spark.read.json("/datasets/amazon/train.json")
train_sample = raw_data.sample(False, 0.005, seed=42).fillna({'reviewText': ''})

# 4. Pipeline
tokenizer = RegexTokenizer(inputCol="reviewText", outputCol="words", pattern="\\W")
swr = StopWordsRemover(inputCol="words", outputCol="filtered_words")
hasher = HashingTF(inputCol="filtered_words", outputCol="features")
lr = LinearRegression(labelCol="overall", featuresCol="features")

pipeline = Pipeline(stages=[tokenizer, swr, hasher, lr])

# 5. Сетка параметров для поиска
paramGrid = (ParamGridBuilder()
             .addGrid(hasher.numFeatures, [1000, 5000]) # Сколько слов учитывать
             .addGrid(lr.regParam, [0.1, 0.01])          # Сила регуляризации
             .build())

# 6. Оценка и Кросс-валидация
evaluator = RegressionEvaluator(labelCol="overall", predictionCol="prediction", metricName="rmse")
cv = CrossValidator(estimator=pipeline, 
                    estimatorParamMaps=paramGrid, 
                    evaluator=evaluator, 
                    numFolds=2)

# 7. Запуск поиска
cv_model = cv.fit(train_sample)

# 8. Вывод результатов
best_model = cv_model.bestModel
print(f"HashingTF numFeatures: {best_model.stages[2].getOrDefault('numFeatures')}")
print(f"LR regParam: {best_model.stages[3].getOrDefault('regParam')}")

# Проверка RMSE
predictions = cv_model.transform(train_sample)
print(f"RMSE на выборке: {evaluator.evaluate(predictions)}")

Picked up _JAVA_OPTIONS: 
Picked up _JAVA_OPTIONS: 
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
                                                                                

HashingTF numFeatures: 1000
LR regParam: 0.1


[Stage 731:====================================================>  (69 + 3) / 72]

RMSE на выборке: 1.0449378867886783


In [2]:
spark.stop()